# Inheritance and Abstract Classes

You will build related classes that share state and behavior while providing their own specialized operations.

An interface gave different classes a common promise. Sometimes related classes also need the same stored information and the same working method. For example, every job in a work plan may have a name, even though different kinds of jobs estimate their duration differently.

You will start with a small visitor-card example to trace how a subclass is constructed and how it uses a method from its base class. Then you will build a named reading job that estimates minutes from a page count. The example will help you distinguish sharing an existing method, replacing a method, and supplying an operation that the base leaves unfinished.

Use the [Module 6 glossary: abstraction, interfaces, and inheritance](terms.md) to review the terms introduced in this lesson.


## Learning Goals

By the end of this lesson, you will be able to:

- Build a subclass that initializes shared state and uses inherited or protected operations.
- Trace an overriding method and an explicit superclass call on the same object.
- Implement and test a concrete subclass that supplies an abstract class's required behavior.


## Why This Matters

Related objects often need a consistent foundation. Work items may all need a name and a description, while each kind uses a different rule to estimate effort. **Inheritance** lets a class build on another class's definition. Shared behavior can stay in one place while the more specific class supplies the parts that differ.

Without that shared foundation, several classes may repeat the same fields and methods. A change then has to be made in several places, and the versions can drift apart. Inheritance is useful only when the more specific class can honor the base class's meaning and operations; similar-looking code alone is not enough reason to form that relationship.

You will connect inheritance to the method selection you practiced with interfaces. The caller's declared type determines which operations it can request, and the actual object determines the overriding instance method that runs. Understanding both parts helps you read class hierarchies, extend existing code, and diagnose initialization or method-selection mistakes.

## Check Your Starting Point

This lesson builds on constructors from Module 5 and interface method selection from the previous notebook. Write your answers below before opening the feedback.

1. In a constructor, what does `this.name = name` copy, and which object receives the value?
2. Why can an object keep a field private while allowing callers to request a public method that uses it?
3. In the previous lesson, a TextProvider variable referred to a FixedText object. Which type determined whether `text()` was an available call, and which object supplied the implementation?
4. What did the `@Override` annotation ask the compiler to check? Would the annotation alone supply a missing method body?

You do not need the new `extends`, `super`, or `abstract` rules to answer these questions.

In [ ]:
My starting-point answers:

[Write your response here. Keep each question or test case clearly labeled.]


<details>
<summary>Show answer</summary>

In the constructor assignment, the right-hand `name` is the argument value received by the parameter. `this.name` is the field of the object being initialized. The assignment stores that value in the current object's field.

Private state and public operations serve different roles. The field holds internal data; the method gives callers a controlled way to request behavior using that data. The caller does not need direct field access to request the method.

The declared TextProvider type made `text()` available because the interface declared it. The FixedText object supplied the implementation that returned its stored value. Assigning the reference did not change the object's class.

`@Override` asked the compiler to verify an implementing or overriding relationship. It did not create an interface relationship or write an implementation. The declaration and method body still had to be correct.

We will now use those same reference and method ideas when one class extends another class.

</details>

## Video Demonstration

Follow the name from the subclass constructor into the abstract base. Then compare the inherited description, the override that extends it, and the required time calculation.

<video controls preload="metadata" width="960">
  <source src="media/02_inheritance_and_abstract_classes/demo.mp4" type="video/mp4">
  <track kind="captions" src="media/02_inheritance_and_abstract_classes/captions.vtt" srclang="en" label="English">
  Your browser does not support embedded video.
</video>

[Read the inheritance and abstract classes demonstration transcript](media/02_inheritance_and_abstract_classes/transcript.md).


## Concept

### Build a more specific class from a shared definition

A campus visitor card needs a person's name and a method that returns that name as a label. Suppose NameCard already supplies those features. VisitorCard can build on that definition instead of repeating the same field and method.

**Class inheritance** lets a more specific class extend another class and inherit eligible members. The class being extended is the **superclass**, also called the base class. The more specific class is the **subclass**. In `class VisitorCard extends NameCard`, the keyword `extends` declares the relationship. A Java class has one direct superclass.

The NameCard constructor receives a name and stores it in a private field. VisitorCard also needs a constructor that accepts a name, because constructors are not inherited. Its `super(name)` statement is an explicit **superclass constructor call**: it supplies the value needed to initialize the base part of this same object.

In this course's Java 21 runtime, that explicit call belongs first in the constructor body. It does not construct a second, separate NameCard object. Run the complete example to create one VisitorCard for Nora and request its label.

In [ ]:
class NameCard {
    private String name;
    public NameCard(String name) {
        this.name = name;
    }
    public String label() {
        return name;
    }
}
class VisitorCard extends NameCard {
    public VisitorCard(String name) {
        super(name);
    }
}
VisitorCard card = new VisitorCard("Nora");
System.out.println(card.label());


The output is `Nora`. The `new VisitorCard("Nora")` expression starts construction of one object. VisitorCard's constructor receives `Nora` and passes it to `super(name)`. The NameCard constructor stores that value in the private field. When construction finishes, the new expression produces the reference assigned to `card`.

VisitorCard has not declared a new `label()` method, so `card.label()` uses the inherited NameCard implementation. That method reads the private field from within NameCard's own code and returns `Nora`. The subclass does not need direct access to the field.

Omitting `super(name)` would not supply the required name. In Java 21, an omitted explicit constructor call implies a no-argument `super()` call, but this NameCard definition has only the constructor that requires a String. The next example keeps the same initialization path and changes the label behavior.

<details class="animation-panel" open>
<summary>One object constructor chain — show or hide animation</summary>
<p><img src="media/02_inheritance_and_abstract_classes/one_object_constructor_chain.gif" alt="new VisitorCard(&quot;Nora&quot;) requests one concrete object. VisitorCard receives Nora and calls super(name) first in this Java 21 constructor. NameCard stores Nora in the private name field of this same object. Both constructor bodies finish. The new expression produces the reference stored in card; construction creates only one VisitorCard object. card.label() uses the inherited method to return Nora; the caller prints Nora." width="960" style="max-width:100%;height:auto;"></p>
</details>

Both constructor bodies initialize one VisitorCard. NameCard writes the private name field, and the caller receives the reference after construction finishes. The inherited label method then reads Nora from that same object. This loop lasts 12.2 seconds.

[View the one object constructor chain still](media/02_inheritance_and_abstract_classes/one_object_constructor_chain_still.png).


### Override a method and reuse its base behavior

The visitor card now has a working label, but staff want it to include the word `visitor`. The existing NameCard method already supplies the person's name. We can use that result as the first part of the more specific label.

**Method overriding** supplies a new implementation of an inherited instance method. VisitorCard declares a public `label()` method with the same name and parameter list as NameCard's method. Its String result agrees with the inherited operation. The familiar `@Override` annotation checks that this relationship is valid.

Inside that override, `super.label()` is a **superclass method call**: it explicitly invokes the superclass's implementation on the current object. This is a method call, not a constructor call. The returned name becomes part of the expression that appends ` visitor`.

The caller below stores a VisitorCard reference in a variable declared as NameCard. It then calls `card.label()` in the ordinary way. Run the example and distinguish that outer call from the explicit `super.label()` inside the override.

In [ ]:
class NameCard {
    private String name;
    public NameCard(String name) {
        this.name = name;
    }
    public String label() {
        return name;
    }
}
class VisitorCard extends NameCard {
    public VisitorCard(String name) {
        super(name);
    }
    @Override
    public String label() {
        return super.label() + " visitor";
    }
}
NameCard card = new VisitorCard("Sam");
System.out.println(card.label());


The output is `Sam visitor`. The actual object is a VisitorCard, even though `card` is declared as NameCard. The ordinary `card.label()` call therefore selects VisitorCard's override, just as an interface-typed call selected an implementing object's method in the previous lesson.

Inside the override, `super.label()` deliberately uses NameCard's method body. That body returns the stored `Sam`. VisitorCard then adds ` visitor` and returns the combined String to the caller, which prints it. Both method bodies operate on the same object's state.

The two call forms serve different purposes: an ordinary instance call selects the overriding implementation for the receiving object, while `super.label()` explicitly reuses the base implementation from within the subclass. Calling `label()` again inside this override would call the same override again instead of obtaining the base result. Next, we will examine a base operation that has no implementation yet.

<details class="animation-panel" open>
<summary>Override and explicit super call — show or hide animation</summary>
<p><img src="media/02_inheritance_and_abstract_classes/override_and_explicit_super_call.gif" alt="card is declared NameCard and refers to VisitorCard initialized with Sam. card.label() selects VisitorCard.label for the actual receiver. super.label() inside the override invokes NameCard.label and returns Sam. The override appends visitor and returns Sam visitor. The caller prints Sam visitor. The explicit base call did not create another object." width="960" style="max-width:100%;height:auto;"></p>
</details>

The base-typed reference reaches a VisitorCard, so the ordinary label call selects its override. The explicit super.label call reuses the base body on the same object. Appending the suffix creates Sam visitor without changing the stored name Sam. This loop lasts 12.2 seconds.

[View the override and explicit super call still](media/02_inheritance_and_abstract_classes/override_and_explicit_super_call_still.png).


### Require a calculation without choosing one for every subclass

A work planner may need an estimated duration from every kind of work. Packing boxes and reading pages use different rules, so a single base calculation would be misleading. The base can require a method while leaving its implementation to the concrete work type.

An **abstract class** is a class that cannot be constructed directly. The keyword `abstract` marks it. It can still have fields, constructors, and working methods. An **abstract method** declares a required operation without a body. Here `public abstract int minutes();` requires an integer duration but supplies no calculation.

A **concrete class** can be instantiated and has implementations for its required abstract operations. PackingWork extends TimedWork and supplies `minutes()`. Its constructor stores a nonnegative box count. For this example, each box takes three minutes, so the method multiplies the number of boxes by three.

The program constructs PackingWork with two boxes and keeps the reference in a TimedWork variable. Run it to see how the shared operation reaches the specific calculation.

In [ ]:
abstract class TimedWork {
    public TimedWork() {
    }
    public abstract int minutes();
}
class PackingWork extends TimedWork {
    private int boxes;
    public PackingWork(int boxes) {
        super();
        this.boxes = boxes;
    }
    @Override
    public int minutes() {
        return boxes * 3;
    }
}
TimedWork work = new PackingWork(2);
System.out.println(work.minutes());


The output is `6`, representing six minutes. PackingWork's constructor stores the value 2 in its `boxes` field. Its explicit no-argument `super()` call participates in initializing the same concrete object; it does not directly instantiate the abstract TimedWork class.

The declared type TimedWork makes `minutes()` available through `work`. The receiving PackingWork object supplies the method body. That body evaluates two boxes times three minutes per box and returns 6, which the caller prints.

An abstract base may leave a required operation unfinished, but a concrete subclass cannot. If PackingWork omitted this implementation, it would have to remain abstract rather than being constructed by this caller. The base states the common requirement, and the concrete class connects it to the appropriate rule and units.

<details class="animation-panel" open>
<summary>Abstract promise concrete behavior — show or hide animation</summary>
<p><img src="media/02_inheritance_and_abstract_classes/abstract_promise_concrete_behavior.gif" alt="TimedWork declares abstract int minutes(); it supplies no calculation body. PackingWork is created with two boxes; its constructor initializes boxes to 2. work is declared TimedWork; work.minutes() selects PackingWork.minutes. Two boxes at three minutes per box gives six minutes. The method returns 6 and the caller prints 6. TimedWork was not directly instantiated." width="960" style="max-width:100%;height:auto;"></p>
</details>

TimedWork states that minutes must be available. The concrete PackingWork object supplies the calculation using its two stored boxes. At three minutes per box, the returned integer 6 represents six minutes; the caller prints that result. This loop lasts 12.2 seconds.

[View the abstract promise concrete behavior still](media/02_inheritance_and_abstract_classes/abstract_promise_concrete_behavior_still.png).


### Offer a narrow operation to subclasses

A subclass sometimes needs shared information without needing direct access to the field that stores it. A heading for a named item, for example, needs to read the name but does not need permission to change it.

**Protected access** permits access within the declaring package and from subclasses under Java's subclass-access rules. A **package** groups related Java types under a name. The keyword `protected` applies this access level to the getter in the example. Protected does not mean “subclasses only”; code in the same package can also have access.

NamedItem keeps `name` private and offers `getName()` as the narrower operation. ItemHeading calls that inherited method from its own `heading()` method and adds the prefix `Item: `. The example uses the getter on the current object. Cross-package access through other object references has additional restrictions that this example does not require.

Run the complete program for an item named Map. Notice how the subclass obtains the name through a method while the field remains private.

In [ ]:
class NamedItem {
    private String name;
    public NamedItem(String name) {
        this.name = name;
    }
    protected String getName() {
        return name;
    }
}
class ItemHeading extends NamedItem {
    public ItemHeading(String name) {
        super(name);
    }
    public String heading() {
        return "Item: " + getName();
    }
}
ItemHeading item = new ItemHeading("Map");
System.out.println(item.heading());


The output is `Item: Map`. ItemHeading's constructor passes `Map` to NamedItem, which stores it in the private field. When `item.heading()` runs, its `getName()` call uses the inherited protected method. That method reads the field from NamedItem's code and returns `Map`; the heading method adds the prefix.

The subclass receives a defined capability instead of direct control over the field. This keeps the base responsible for its representation. A protected field would expose more implementation detail; the access modifier alone would not make that field a good design choice.

The worked example will combine these mechanisms. A base will store a job name, offer a protected getter, provide a description, and require a duration calculation. A subclass will initialize that shared state, extend the description, and supply the calculation.

### Combine the Mechanisms Deliberately

The next program gives every named job a common foundation while letting reading work supply its own duration rule. Use the table to connect each Java mechanism to its job in the design.

| Mechanism | Purpose |
|---|---|
| `extends NamedJob` | Establish the subclass relationship. |
| `super(name)` | Initialize the shared name in the same object. |
| `abstract int minutes()` | Require a concrete duration calculation. |
| `@Override` | Check an intended method implementation or replacement. |
| `super.description()` | Reuse the base description inside the override. |
| `protected getName()` | Provide a narrow operation for subclass code. |
| A NamedJob reference | Expose base operations while the object supplies specialized behavior. |

A method specific to ReadingJob is not automatically available through a NamedJob variable. The caller will keep a ReadingJob reference for its extra heading operation and use the base reference for the shared operations.

These choices should express a meaningful relationship, not simply remove repeated text. Each subclass must honor the base operations. This example keeps the hierarchy one level deep and initializes fields directly in constructors; it does not call overridable methods during construction.

## Worked Example

### Describe a Reading Job and Estimate Its Duration

A coordinator is planning a reading task named `Guide`. The task has three pages. For this example, the planning rule is two minutes per page, and page counts must be nonnegative whole numbers. The coordinator needs three report lines: a heading with the job name, a description identifying reading work, and an estimated duration labeled in minutes.

NamedJob will store the shared name in a private field. Its constructor receives the name, its protected getter returns it to subclass code, and its public description method provides the basic text. It declares `minutes()` as abstract because different job types need different duration rules.

ReadingJob adds a private page count. Its constructor calls `super(name)` first to initialize the shared name, then stores `pages` in the same object. Its description override calls `super.description()` and adds ` reading`. Its minutes implementation multiplies the page count by two. Finally, its heading method uses the protected getter and adds `Job: `.

The caller constructs one ReadingJob and stores its reference in `reading`. Assigning that reference to `NamedJob job` creates a second reference variable, not a second job. The heading call uses `reading` because heading is a ReadingJob-specific operation. The description and minutes calls use `job` because both operations are available through NamedJob. Run the program to trace construction and then each report call.

In [ ]:
abstract class NamedJob {
    private String name;
    public NamedJob(String name) {
        this.name = name;
    }
    protected String getName() {
        return name;
    }
    public String description() {
        return getName();
    }
    public abstract int minutes();
}
class ReadingJob extends NamedJob {
    private int pages;
    public ReadingJob(String name, int pages) {
        super(name);
        this.pages = pages;
    }
    @Override
    public String description() {
        return super.description() + " reading";
    }
    @Override
    public int minutes() {
        return pages * 2;
    }
    public String heading() {
        return "Job: " + getName();
    }
}
ReadingJob reading = new ReadingJob("Guide", 3);
NamedJob job = reading;
System.out.println(reading.heading());
System.out.println(job.description());
System.out.println("Minutes: " + job.minutes());


The program prints:

```text
Job: Guide
Guide reading
Minutes: 6
```

Construction places `Guide` in the base class's private name field and 3 in the subclass's private page field. Both fields belong to the same ReadingJob object. The first report calls `reading.heading()`. That method obtains the name through `getName()` and returns `Job: Guide`.

The second report calls `job.description()`. Although `job` is declared as NamedJob, its reference reaches a ReadingJob, so the override runs. Inside that override, `super.description()` explicitly reuses the base method to obtain `Guide`. The override appends ` reading`, yielding the second line.

The final call reaches ReadingJob's implementation of the abstract minutes operation. Three pages at two minutes per page gives six minutes. The caller adds the `Minutes: ` label when printing the result.

The abstract base can define the operations used through `job` without being directly constructible. The specific object supplies the missing calculation. A call to `job.heading()` would not be available through the declared NamedJob type, even though the receiving object has that method. Keep the distinction between the caller's available operations and the object's implementation as you begin practice.

## Guided Practice

### Predict a New Reading Job

The complete program below constructs ReadingJob("Manual", 5). The name is text, the page count is five, and the rule remains two minutes per page. Before running, predict all three report lines in the response cell.

Trace where the constructor stores the name and page count. Then name the method body reached by each report call, including the explicit `super.description()` call inside the override. Preserve your prediction when you run the program.

In [ ]:
My prediction and reasoning:

[Write your response here. Keep each question or test case clearly labeled.]


In [ ]:
abstract class NamedJob {
    private String name;
    public NamedJob(String name) {
        this.name = name;
    }
    protected String getName() {
        return name;
    }
    public String description() {
        return getName();
    }
    public abstract int minutes();
}
class ReadingJob extends NamedJob {
    private int pages;
    public ReadingJob(String name, int pages) {
        super(name);
        this.pages = pages;
    }
    @Override
    public String description() {
        return super.description() + " reading";
    }
    @Override
    public int minutes() {
        return pages * 2;
    }
    public String heading() {
        return "Job: " + getName();
    }
}
ReadingJob reading = new ReadingJob("Manual", 5);
NamedJob job = reading;
System.out.println(reading.heading());
System.out.println(job.description());
System.out.println("Minutes: " + job.minutes());


Record the actual three lines and compare them with your prediction. If they differ, identify whether the mistake involved initialization, ordinary method selection, the explicit base-method call, or the duration calculation. Explain the corrected step.

In [ ]:
My observed output and comparison:

[Write your response here. Keep each question or test case clearly labeled.]


Trace construction and report calls in the response cell. Explain how `super(name)` differs from `super.description()`, why the program has one job but two reference variables, and how the protected getter differs from direct private-field access.

Also diagnose these proposed alternatives without running them: directly constructing NamedJob, or calling `job.heading()` through the NamedJob variable. Identify the declaration that makes each alternative unavailable. Finish by distinguishing the required minutes operation from its concrete implementation.

In [ ]:
My step-by-step trace and explanation:

[Write your response here. Keep each question or test case clearly labeled.]


<details>
<summary>Show answer</summary>

The new expression creates one ReadingJob. ReadingJob first calls super(name), so NamedJob stores Manual in its private name field. The subclass then stores 5 in pages.

The variable `reading` has declared type ReadingJob; `job` has declared type NamedJob, and both refer to that same object. reading.heading() uses the inherited protected getName() to obtain Manual. job.description() selects ReadingJob.description; its super.description() call deliberately uses NamedJob.description, which obtains the name, before the subclass appends the text ` reading`. job.minutes() selects ReadingJob.minutes and returns 5 * 2 = 10.

The abstract NamedJob type can declare the reference even though NamedJob cannot itself be constructed. NamedJob.description has a body; NamedJob.minutes is abstract and ends with a semicolon. ReadingJob supplies minutes and replaces description.

Constructors are not inherited: ReadingJob declares its own constructor and calls the named base constructor.

The protected getter is used inside the subclass on the current object while the underlying name stays private. Protected access also includes the declaring package; it does not mean subclasses only. job.heading() is unavailable through the NamedJob declared type, and direct construction of NamedJob is disallowed because the class is abstract.

```java
abstract class NamedJob {
    private String name;
    public NamedJob(String name) {
        this.name = name;
    }
    protected String getName() {
        return name;
    }
    public String description() {
        return getName();
    }
    public abstract int minutes();
}
class ReadingJob extends NamedJob {
    private int pages;
    public ReadingJob(String name, int pages) {
        super(name);
        this.pages = pages;
    }
    @Override
    public String description() {
        return super.description() + " reading";
    }
    @Override
    public int minutes() {
        return pages * 2;
    }
    public String heading() {
        return "Job: " + getName();
    }
}
ReadingJob reading = new ReadingJob("Manual", 5);
NamedJob job = reading;
System.out.println(reading.heading());
System.out.println(job.description());
System.out.println("Minutes: " + job.minutes());
```

Expected output:

```text
Job: Manual
Manual reading
Minutes: 10
```

Common error: Treating super(name) as a second new object. Using the base description alone for the call on a ReadingJob object. Predicting the earlier Guide and 3 values instead of these constructor inputs.

</details>

### Inherit One Method and Implement Another

The complete QuickJob example below uses the name Check. QuickJob supplies a minutes implementation but does not replace description. Before running, predict the two printed lines and identify the body each call should use.

Count the object constructions and explain how the abstract base's constructor participates. Compare the planned description call with the earlier ReadingJob override. Record your reasoning in the response cell, then run the complete example.

In [ ]:
My prediction and reasoning:

[Write your response here. Keep each question or test case clearly labeled.]


In [ ]:
abstract class NamedJob {
    private String name;
    public NamedJob(String name) {
        this.name = name;
    }
    protected String getName() {
        return name;
    }
    public String description() {
        return getName();
    }
    public abstract int minutes();
}
class QuickJob extends NamedJob {
    public QuickJob(String name) {
        super(name);
    }
    @Override
    public int minutes() {
        return 1;
    }
}
NamedJob job = new QuickJob("Check");
System.out.println(job.description());
System.out.println("Minutes: " + job.minutes());


Record the two actual lines and compare them with your prediction. Explain why QuickJob inherits the base description while ReadingJob supplied an override.

As a written diagnosis only, consider deleting QuickJob's minutes implementation. Would the class still be complete enough to construct as a concrete class? Explain which required operation would remain missing. Do not execute the incomplete variant.

In [ ]:
My observed results, comparison, and explanation:

[Write your response here. Keep each question or test case clearly labeled.]


<details>
<summary>Show answer</summary>

QuickJob extends NamedJob and calls super(name), so NamedJob initializes the private name as Check inside the one QuickJob object. QuickJob does not override description, so its public inherited NamedJob.description body runs and returns the name through getName(). QuickJob implements the abstract minutes operation with a return value of 1. A NamedJob reference can invoke both operations. Deleting that implementation would leave the declared concrete QuickJob without the required minutes body; it would not be a complete concrete implementation. An abstract class can have a constructor used during subclass construction even though it cannot be directly instantiated.

```java
abstract class NamedJob {
    private String name;
    public NamedJob(String name) {
        this.name = name;
    }
    protected String getName() {
        return name;
    }
    public String description() {
        return getName();
    }
    public abstract int minutes();
}
class QuickJob extends NamedJob {
    public QuickJob(String name) {
        super(name);
    }
    @Override
    public int minutes() {
        return 1;
    }
}
NamedJob job = new QuickJob("Check");
System.out.println(job.description());
System.out.println("Minutes: " + job.minutes());
```

Expected output:

```text
Check
Minutes: 1
```

Common error: Assuming every inherited method must be rewritten in the subclass. Assuming an abstract class cannot contain a constructor or implemented method. Removing the abstract requirement because an unrelated method was inherited.

</details>

### Complete the Subclass Connections

A sorting job named Parcel handles two items at three minutes per item. The displayed draft needs four connections between SortingJob and NamedJob. It is incomplete and must not be run as written.

Choose replacements for RELATION, BASE_INIT, BASE_DESCRIPTION, and NAME_READER using `extends`, `super(name)`, `super.description()`, and `getName()`, each once. In the response cell, predict the three output lines and explain what each replacement connects.

Then copy the complete draft into the Java work cell, make the four replacements, and run it. Keep the base constructor call first, preserve the item count, and leave all other code unchanged.

```java
abstract class NamedJob {
    private String name;
    public NamedJob(String name) {
        this.name = name;
    }
    protected String getName() {
        return name;
    }
    public String description() {
        return getName();
    }
    public abstract int minutes();
}
class SortingJob RELATION NamedJob {
    private int items;
    public SortingJob(String name, int items) {
        BASE_INIT;
        this.items = items;
    }
    @Override
    public String description() {
        return BASE_DESCRIPTION + " sorting";
    }
    @Override
    public int minutes() {
        return items * 3;
    }
    public String heading() {
        return "Job: " + NAME_READER;
    }
}
SortingJob sorting = new SortingJob("Parcel", 2);
NamedJob job = sorting;
System.out.println(sorting.heading());
System.out.println(job.description());
System.out.println("Minutes: " + job.minutes());
```

In [ ]:
My plan, predicted results, and reasoning:

[Write your response here. Keep each question or test case clearly labeled.]


Record all three actual lines and compare them with your prediction. Explain the order of base-name and subclass-item initialization, the name returned by the explicit base description call, and the protected getter used by heading. State why this class relationship uses `extends`.

In [ ]:
My observed results, comparison, and explanation:

[Write your response here. Keep each question or test case clearly labeled.]


<details>
<summary>Show answer</summary>

RELATION is extends, BASE_INIT is super(name), BASE_DESCRIPTION is super.description(), and NAME_READER is getName().

SortingJob is a subclass of NamedJob. Its constructor first supplies Parcel to the base constructor, then stores 2 in items.

The heading uses the inherited protected getter. The description override reuses the base’s Parcel result before adding sorting. The required minutes implementation returns 2 * 3 = 6. The NamedJob reference permits description and minutes, while the SortingJob reference permits its additional heading.

```java
abstract class NamedJob {
    private String name;
    public NamedJob(String name) {
        this.name = name;
    }
    protected String getName() {
        return name;
    }
    public String description() {
        return getName();
    }
    public abstract int minutes();
}
class SortingJob extends NamedJob {
    private int items;
    public SortingJob(String name, int items) {
        super(name);
        this.items = items;
    }
    @Override
    public String description() {
        return super.description() + " sorting";
    }
    @Override
    public int minutes() {
        return items * 3;
    }
    public String heading() {
        return "Job: " + getName();
    }
}
SortingJob sorting = new SortingJob("Parcel", 2);
NamedJob job = sorting;
System.out.println(sorting.heading());
System.out.println(job.description());
System.out.println("Minutes: " + job.minutes());
```

Expected output:

```text
Job: Parcel
Parcel sorting
Minutes: 6
```

Common error: Using implements for this class-to-class relationship. Passing no name to a base constructor that requires one. Calling description() again instead of the intended superclass body. Trying to use the private name field in the heading.

</details>

### Change an Override While Reusing the Base Result

The coordinator wants the type of work before its name in the description. Start with the complete Manual/5 prediction program. Change only ReadingJob.description so it returns `"Reading: " + super.description()`.

Keep the constructors, heading, minutes, base class, and caller unchanged. Predict all three lines in the response cell and explain which line should change. Then copy the complete modified program into the Java work cell and run it. Keep the explicit superclass call; calling `description()` again on the current object would re-enter the override.

In [ ]:
My prediction and reasoning:

[Write your response here. Keep each question or test case clearly labeled.]


In [ ]:
abstract class NamedJob {
    private String name;
    public NamedJob(String name) {
        this.name = name;
    }
    protected String getName() {
        return name;
    }
    public String description() {
        return getName();
    }
    public abstract int minutes();
}
class ReadingJob extends NamedJob {
    private int pages;
    public ReadingJob(String name, int pages) {
        super(name);
        this.pages = pages;
    }
    @Override
    public String description() {
        return super.description() + " reading";
    }
    @Override
    public int minutes() {
        return pages * 2;
    }
    public String heading() {
        return "Job: " + getName();
    }
}
ReadingJob reading = new ReadingJob("Manual", 5);
NamedJob job = reading;
System.out.println(reading.heading());
System.out.println(job.description());
System.out.println("Minutes: " + job.minutes());


Record the modified output and compare it with your prediction. Explain why the description changes while the heading and duration stay the same. Include why the call through a NamedJob reference still selects ReadingJob's override.

In [ ]:
My observed results, comparison, and explanation:

[Write your response here. Keep each question or test case clearly labeled.]


Next, change only the constructor inputs to `Outline` and 0. Keep the modified description method. Predict the three output lines before running. Explain which results depend on the name and which depend on the page count.

In [ ]:
My prediction and reasoning:

[Write your response here. Keep each question or test case clearly labeled.]


Return to the complete modification program and make only the Outline/0 input change. Run it after recording the variant prediction.

Record the Outline/0 output and compare it with your prediction. Explain how a named job can have an estimated duration of zero. Restore Manual and 5, run again, and record the restored output before opening the answer.

In [ ]:
My observed results, comparison, and explanation:

[Write your response here. Keep each question or test case clearly labeled.]


<details>
<summary>Show answer</summary>

The call job.description() still selects ReadingJob.description because the actual object is a ReadingJob. Inside that override, super.description() supplies the base result Manual; the concatenation places the text `Reading: ` before that result. The constructor state, heading and minutes implementation are unchanged, so the first and third lines remain Job: Manual and Minutes: 10.

With Outline and 0, the shared name changes both text results while zero pages produce zero minutes. A plain description() call from inside this override would call the override again instead of selecting the base body.

```java
abstract class NamedJob {
    private String name;
    public NamedJob(String name) {
        this.name = name;
    }
    protected String getName() {
        return name;
    }
    public String description() {
        return getName();
    }
    public abstract int minutes();
}
class ReadingJob extends NamedJob {
    private int pages;
    public ReadingJob(String name, int pages) {
        super(name);
        this.pages = pages;
    }
    @Override
    public String description() {
        return "Reading: " + super.description();
    }
    @Override
    public int minutes() {
        return pages * 2;
    }
    public String heading() {
        return "Job: " + getName();
    }
}
ReadingJob reading = new ReadingJob("Manual", 5);
NamedJob job = reading;
System.out.println(reading.heading());
System.out.println(job.description());
System.out.println("Minutes: " + job.minutes());
```

Expected output:

```text
Job: Manual
Reading: Manual
Minutes: 10
```

Common error: Changing the base description and affecting the wrong part of the design. Hard-coding Manual instead of using the base result. Replacing the explicit superclass call with another call to the override. Changing minutes even though the task changes only the description.

**Additional test: `Modified description with Outline and zero pages`.** The name travels through the base constructor and getter; the zero page count travels through the subclass constructor and calculation. The changed input also catches a hard-coded Manual description.

```java
abstract class NamedJob {
    private String name;
    public NamedJob(String name) {
        this.name = name;
    }
    protected String getName() {
        return name;
    }
    public String description() {
        return getName();
    }
    public abstract int minutes();
}
class ReadingJob extends NamedJob {
    private int pages;
    public ReadingJob(String name, int pages) {
        super(name);
        this.pages = pages;
    }
    @Override
    public String description() {
        return "Reading: " + super.description();
    }
    @Override
    public int minutes() {
        return pages * 2;
    }
    public String heading() {
        return "Job: " + getName();
    }
}
ReadingJob reading = new ReadingJob("Outline", 0);
NamedJob job = reading;
System.out.println(reading.heading());
System.out.println(job.description());
System.out.println("Minutes: " + job.minutes());
```

Expected output:

```text
Job: Outline
Reading: Outline
Minutes: 0
```

</details>

### Repair the Superclass Constructor Order

The displayed program uses Guide and three pages, but its ReadingJob constructor puts the superclass call after a field assignment. That order is invalid in this course's Java 21 runtime. Do not run the draft as written.

Identify the two statements that need to exchange positions. In the response cell, explain why `super(name)` must come first and predict the repaired report. Then copy the complete program into the Java work cell, move that call before the page-field assignment, and run the repair. Preserve both statement contents and all other code.

```java
abstract class NamedJob {
    private String name;
    public NamedJob(String name) {
        this.name = name;
    }
    protected String getName() {
        return name;
    }
    public String description() {
        return getName();
    }
    public abstract int minutes();
}
class ReadingJob extends NamedJob {
    private int pages;
    public ReadingJob(String name, int pages) {
        this.pages = pages;
        super(name);
    }
    @Override
    public String description() {
        return super.description() + " reading";
    }
    @Override
    public int minutes() {
        return pages * 2;
    }
    public String heading() {
        return "Job: " + getName();
    }
}
ReadingJob reading = new ReadingJob("Guide", 3);
NamedJob job = reading;
System.out.println(reading.heading());
System.out.println(job.description());
System.out.println("Minutes: " + job.minutes());
```

In [ ]:
My plan, predicted results, and reasoning:

[Write your response here. Keep each question or test case clearly labeled.]


Record the repaired output and compare it with your prediction. Explain why deleting `super(name)` would not supply the required base name. Also explain why constructors are not inherited and why the two constructor bodies initialize one object rather than two.

In [ ]:
My observed results, comparison, and explanation:

[Write your response here. Keep each question or test case clearly labeled.]


<details>
<summary>Show answer</summary>

The explicit super(name) call must be first in this Java 21 constructor. It initializes the NamedJob portion with Guide before ReadingJob stores 3 in pages. Moving the call, while keeping both statements, restores the original canonical program.

Omitting it would not satisfy NamedJob’s constructor that requires a String argument. ReadingJob declares its own constructor; it does not inherit NamedJob’s constructor.

The one new ReadingJob expression creates one object whose base and subclass portions are initialized in order. Its heading and description use Guide, and its minutes result is 3 * 2 = 6.

```java
abstract class NamedJob {
    private String name;
    public NamedJob(String name) {
        this.name = name;
    }
    protected String getName() {
        return name;
    }
    public String description() {
        return getName();
    }
    public abstract int minutes();
}
class ReadingJob extends NamedJob {
    private int pages;
    public ReadingJob(String name, int pages) {
        super(name);
        this.pages = pages;
    }
    @Override
    public String description() {
        return super.description() + " reading";
    }
    @Override
    public int minutes() {
        return pages * 2;
    }
    public String heading() {
        return "Job: " + getName();
    }
}
ReadingJob reading = new ReadingJob("Guide", 3);
NamedJob job = reading;
System.out.println(reading.heading());
System.out.println(job.description());
System.out.println("Minutes: " + job.minutes());
```

Expected output:

```text
Job: Guide
Guide reading
Minutes: 6
```

Common error: Deleting the superclass call rather than placing it correctly. Moving the field assignment into the base class and changing the contract. Making the private base name public as an unrelated workaround. Claiming super(name) constructs a second object.

</details>

## Independent Practice

### Build a Concrete Job from the Abstract Base

A coordinator needs a build task named Lab with four steps. The classroom estimate is five minutes per step. Steps are nonnegative whole numbers, and the required reports are a task heading, a build description, and labeled minutes.

Keep the supplied NamedJob definition unchanged. Add BuildJob with a private steps field. Its constructor must call `super(name)` first, then store steps. Implement minutes as steps times five. Override description to append ` build` to `super.description()`. Add heading to return `Task: ` followed by the protected getter's result.

Construct BuildJob("Lab", 4), keep a BuildJob reference for heading, and assign the same reference to `NamedJob job` for description and minutes. Print the heading first, then the description, then `Minutes: ` followed by the duration. Use public methods and `@Override` for the required or replaced operations.

Before writing, predict the three output lines and describe the state and method relationships in the response cell. Copy the supplied base into the Java work cell, then add your subclass and complete caller.

```java
abstract class NamedJob {
    private String name;
    public NamedJob(String name) {
        this.name = name;
    }
    protected String getName() {
        return name;
    }
    public String description() {
        return getName();
    }
    public abstract int minutes();
}
```

In [ ]:
My plan, predicted results, and reasoning:

[Write your response here. Keep each question or test case clearly labeled.]


Record the three actual lines and compare them with your prediction. Trace the name and step values through construction, then trace the body reached by each printed call. Explain how the protected getter supports the subclass while the field stays private.

Explain why direct NamedJob construction and `job.heading()` are not valid alternatives. Keep those alternatives as written diagnoses, not executable tests.

In [ ]:
My observed results, comparison, and explanation:

[Write your response here. Keep each question or test case clearly labeled.]


### Test Counts and Names Separately

Use the complete BuildJob program for each case. Keep the base and subclass definitions unchanged. Test Lab/4, Lab/0, Lab/1, and Annex/2 as name/step inputs, then restore Lab/4.

Before each run, record the case label and all three predicted report lines in the response cell. Explain which lines should respond to a name change and which should respond to the step count. Preserve each prediction for comparison.

In [ ]:
My prediction and reasoning:

[Write your response here. Keep each question or test case clearly labeled.]


For each case, record its prediction first, then change only the constructor inputs in your complete independent program and run it. Record the actual output below before predicting the next case. Finish with the restored Lab/4 program.

Record each case's actual three lines and compare them with the saved prediction. Explain what the zero-step and one-step cases test, why changing the name does not change the rate, and which method bodies remain the same across cases.

If you repair the program, describe the cause and rerun the affected cases. Record the restored Lab/4 output at the end.

In [ ]:
My observed results, comparison, and explanation:

[Write your response here. Keep each question or test case clearly labeled.]


<details>
<summary>Show answer</summary>

BuildJob extends NamedJob. Its constructor first passes Lab to the superclass constructor, which stores the private name, then stores 4 in the private steps field. Its minutes implementation supplies the abstract requirement and returns 4 * 5 = 20. Its description override calls super.description(), receives Lab, and appends build. Its heading uses the inherited protected getName() to build Task: Lab.

building and job refer to the same BuildJob object. The BuildJob reference exposes heading; the NamedJob reference exposes description and minutes, whose calls select the BuildJob implementations. NamedJob cannot be directly constructed because it is abstract, and job.heading() is unavailable through its declared type.

Counts are nonnegative classroom integers with products fitting int; these implementations rely on that input rule and do not validate it.

For Lab/0, the constructors still create a valid concrete object with the name Lab; the two text lines stay Task: Lab and Lab build, while steps * 5 gives zero. Lab/1 yields five minutes and checks the single-step rule. Annex/2 must change both name-based lines and yield ten minutes. All methods still act on one BuildJob through the appropriate declared reference.

These tests separate shared name state from subclass numeric state and catch hard-coded results. They do not make NamedJob constructible or add heading to the NamedJob declared type. Recreate the complete object for each case and restore the baseline afterward.

```java
abstract class NamedJob {
    private String name;
    public NamedJob(String name) {
        this.name = name;
    }
    protected String getName() {
        return name;
    }
    public String description() {
        return getName();
    }
    public abstract int minutes();
}
class BuildJob extends NamedJob {
    private int steps;
    public BuildJob(String name, int steps) {
        super(name);
        this.steps = steps;
    }
    @Override
    public String description() {
        return super.description() + " build";
    }
    @Override
    public int minutes() {
        return steps * 5;
    }
    public String heading() {
        return "Task: " + getName();
    }
}
BuildJob building = new BuildJob("Lab", 4);
NamedJob job = building;
System.out.println(building.heading());
System.out.println(job.description());
System.out.println("Minutes: " + job.minutes());
```

Expected output:

```text
Task: Lab
Lab build
Minutes: 20
```

Common error: Changing the supplied abstract base instead of implementing its requirement. Reading the private name directly in BuildJob. Giving minutes a changed method name, parameter list, or result type or omitting its body. Calling heading through the NamedJob reference. Hard-coding the name or step count instead of using constructor state.

**Additional test: Lab with zero steps.** The name is initialized normally and both text operations still work. Zero is a valid step count and produces zero minutes.

```java
abstract class NamedJob {
    private String name;
    public NamedJob(String name) {
        this.name = name;
    }
    protected String getName() {
        return name;
    }
    public String description() {
        return getName();
    }
    public abstract int minutes();
}
class BuildJob extends NamedJob {
    private int steps;
    public BuildJob(String name, int steps) {
        super(name);
        this.steps = steps;
    }
    @Override
    public String description() {
        return super.description() + " build";
    }
    @Override
    public int minutes() {
        return steps * 5;
    }
    public String heading() {
        return "Task: " + getName();
    }
}
BuildJob building = new BuildJob("Lab", 0);
NamedJob job = building;
System.out.println(building.heading());
System.out.println(job.description());
System.out.println("Minutes: " + job.minutes());
```

Expected output:

```text
Task: Lab
Lab build
Minutes: 0
```

**Additional test: Lab with one step.** One step produces five minutes while the shared name-based behavior remains unchanged.

```java
abstract class NamedJob {
    private String name;
    public NamedJob(String name) {
        this.name = name;
    }
    protected String getName() {
        return name;
    }
    public String description() {
        return getName();
    }
    public abstract int minutes();
}
class BuildJob extends NamedJob {
    private int steps;
    public BuildJob(String name, int steps) {
        super(name);
        this.steps = steps;
    }
    @Override
    public String description() {
        return super.description() + " build";
    }
    @Override
    public int minutes() {
        return steps * 5;
    }
    public String heading() {
        return "Task: " + getName();
    }
}
BuildJob building = new BuildJob("Lab", 1);
NamedJob job = building;
System.out.println(building.heading());
System.out.println(job.description());
System.out.println("Minutes: " + job.minutes());
```

Expected output:

```text
Task: Lab
Lab build
Minutes: 5
```

**Additional test: Annex with two steps.** Both heading and description must use Annex from the base state, and the subclass must compute 2 * 5 rather than a hard-coded baseline result.

```java
abstract class NamedJob {
    private String name;
    public NamedJob(String name) {
        this.name = name;
    }
    protected String getName() {
        return name;
    }
    public String description() {
        return getName();
    }
    public abstract int minutes();
}
class BuildJob extends NamedJob {
    private int steps;
    public BuildJob(String name, int steps) {
        super(name);
        this.steps = steps;
    }
    @Override
    public String description() {
        return super.description() + " build";
    }
    @Override
    public int minutes() {
        return steps * 5;
    }
    public String heading() {
        return "Task: " + getName();
    }
}
BuildJob building = new BuildJob("Annex", 2);
NamedJob job = building;
System.out.println(building.heading());
System.out.println(job.description());
System.out.println("Minutes: " + job.minutes());
```

Expected output:

```text
Task: Annex
Annex build
Minutes: 10
```

</details>

## Summary

A subclass extends one direct superclass and can inherit eligible behavior. Its constructor must arrange the initialization required by the superclass. In this lesson's Java 21 examples, the explicit `super(...)` call comes first and initializes the base portion of the same object.

An ordinary instance call can select the subclass's override through a base-typed reference. An explicit `super.method()` call inside the subclass instead reuses the superclass implementation on that object. These calls serve different purposes; neither constructs an extra object.

An abstract class can combine state and working methods with required abstract operations. A concrete subclass supplies the missing behavior. Protected operations can give subclass code a narrow capability while fields remain private, subject to Java's package and subclass-access rules.

### Recall Construction and Method Calls

In the response cell, explain the difference between `super(name)` in a constructor and `super.description()` inside an overriding method. State what each call accomplishes and whether either creates a second object.

Then trace an ordinary `job.description()` call when `job` is declared as the base type but refers to an object with an overriding subclass method. Explain why this call differs from the explicit superclass method call.

In [ ]:
My explanation from memory:

[Write your response here. Keep each question or test case clearly labeled.]


## Reflection

### Transfer the Idea

Choose two real kinds of work that share a name and description but estimate effort using different rules. In the response cell, describe what a common base class could store and implement, and which operation it should leave abstract.

State the units for each effort rule and give one example input and expected result for each kind of work. Explain the common promise both subclasses must honor so a caller can treat either as the base type.

In [ ]:
My transfer example and explanation:

[Write your response here. Keep each question or test case clearly labeled.]


## Supplemental Reading

- [Java class inheritance](https://dev.java/learn/inheritance/what-is-inheritance/) — Introduces superclass and subclass relationships and inherited members.

- [Abstract methods and classes](https://dev.java/learn/inheritance/abstract-classes/) — Explains abstract classes, required abstract methods, and concrete subclasses.

- [Java 21 class rules](https://docs.oracle.com/javase/specs/jls/se21/html/jls-8.html) — Provides the Java 21 rules for classes, inheritance, methods, and constructor bodies.

- [Java 21 access control](https://docs.oracle.com/javase/specs/jls/se21/html/jls-6.html#jls-6.6) — Specifies public, protected, package, and private access rules.